# 04 — Rhetoric-Action Gap Analysis

Connects the NLP rhetoric scores to arms transfer behavior from the companion network pipeline.
Countries above the diagonal say more than they do (hypocrites); below = quiet good actors.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT = Path('../output')
METRICS = OUTPUT / 'metrics'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print('Output dir:', METRICS)

In [ ]:
# Load core gap data
gap_df = pd.read_csv(METRICS / 'rhetoric_action_gap.csv')
rhetoric_df = pd.read_csv(METRICS / 'rhetoric_scores.csv')
action_df = pd.read_csv(METRICS / 'action_scores.csv')

print(f'Gap data: {len(gap_df)} rows, {gap_df["country_code"].nunique()} countries')
print(f'Years: {gap_df["year"].min()} – {gap_df["year"].max()}')
gap_df.head()

In [ ]:
# Mean gap per country (averaged across all years)
mean_gap = (
    gap_df.groupby('country_code')[['rhetoric_score', 'action_score', 'gap']]
    .mean()
    .sort_values('gap', ascending=False)
)

print('Top 10 hypocrites (high rhetoric, poor behavior):')
print(mean_gap.head(10).to_string())
print()
print('Top 10 quiet good actors (low rhetoric, responsible behavior):')
print(mean_gap.tail(10).to_string())

In [ ]:
# Rhetoric vs action scatter — the main gap plot
fig, ax = plt.subplots(figsize=(12, 9))

year_sample = gap_df[gap_df['year'] == 2020].copy()

ax.scatter(
    year_sample['rhetoric_score'],
    year_sample['action_score'],
    alpha=0.7, s=60, color='steelblue', edgecolors='white', linewidth=0.5
)

# Diagonal = perfect alignment
lims = [0, 1]
ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1, label='Perfect alignment')

# Label notable countries
notable = ['USA', 'RUS', 'CHN', 'FRA', 'GBR', 'DEU', 'SAU', 'IND', 'PAK', 'AUT', 'NZL', 'MEX']
for _, row in year_sample[year_sample['country_code'].isin(notable)].iterrows():
    ax.annotate(row['country_code'], (row['rhetoric_score'], row['action_score']),
                fontsize=8, ha='left', xytext=(4, 2), textcoords='offset points')

ax.fill_between(lims, lims, [1, 1], alpha=0.05, color='red', label='Hypocrite zone (rhetoric > action)')
ax.fill_between(lims, [0, 0], lims, alpha=0.05, color='green', label='Quiet actor zone (action > rhetoric)')

ax.set_xlabel('Rhetoric Score (NLP composite)', fontsize=12)
ax.set_ylabel('Action Score (behavior composite)', fontsize=12)
ax.set_title('Rhetoric-Action Gap — 2020', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES / 'gap_scatter_2020.png', dpi=150)
plt.show()

In [ ]:
# Gap ranking bar chart for 2020
year_mean = gap_df[gap_df['year'] == 2020].sort_values('gap', ascending=True)
top30 = pd.concat([year_mean.head(15), year_mean.tail(15)])

colors = ['#d73027' if g > 0 else '#4575b4' for g in top30['gap']]

fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(top30['country_code'], top30['gap'], color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Rhetoric-Action Gap', fontsize=11)
ax.set_title('Rhetoric-Action Gap Ranking (2020)\nRed = rhetoric > behavior | Blue = behavior > rhetoric', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'gap_ranking_2020.png', dpi=150)
plt.show()

In [ ]:
# Gap time series for major exporters
major_exporters = ['USA', 'RUS', 'FRA', 'GBR', 'DEU', 'CHN']
ts = gap_df[gap_df['country_code'].isin(major_exporters)]

fig, ax = plt.subplots(figsize=(13, 6))
for country, grp in ts.groupby('country_code'):
    grp = grp.sort_values('year')
    ax.plot(grp['year'], grp['gap'], marker='o', markersize=3, label=country, linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.fill_between(gap_df['year'].unique(), 0, 1, alpha=0.03, color='red')
ax.fill_between(gap_df['year'].unique(), -1, 0, alpha=0.03, color='blue')
ax.set_xlabel('Year'); ax.set_ylabel('Rhetoric-Action Gap')
ax.set_title('Rhetoric-Action Gap Over Time — Major Arms Exporters', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'gap_time_series_exporters.png', dpi=150)
plt.show()

In [ ]:
# Distribution of gap by gap_category
if 'gap_category' in gap_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_cat = {'hypocrite': '#d73027', 'aligned': '#fee08b', 'quiet_good_actor': '#4575b4'}
    for cat, grp in gap_df.groupby('gap_category'):
        ax.hist(grp['gap'], bins=40, alpha=0.6, label=cat, color=colors_cat.get(cat, 'grey'))
    ax.set_xlabel('Rhetoric-Action Gap'); ax.set_ylabel('Count')
    ax.set_title('Distribution of Rhetoric-Action Gap by Category')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / 'gap_distribution.png', dpi=150)
    plt.show()

    # Category counts
    print(gap_df['gap_category'].value_counts())

In [ ]:
# Rhetoric score components breakdown for selected countries
if rhetoric_df is not None:
    cols = [c for c in rhetoric_df.columns if c not in ['country_code', 'year', 'rhetoric_score']]
    selected = ['USA', 'RUS', 'FRA', 'DEU', 'CHN', 'SAU', 'AUT', 'NZL']
    sub = rhetoric_df[
        (rhetoric_df['country_code'].isin(selected)) &
        (rhetoric_df['year'] == 2020)
    ].set_index('country_code')

    if cols:
        ax = sub[cols].plot(kind='bar', figsize=(13, 5), colormap='tab10', edgecolor='white')
        ax.set_title('Rhetoric Score Components by Country (2020)', fontsize=12)
        ax.set_xlabel('')
        ax.legend(bbox_to_anchor=(1.01, 1))
        plt.tight_layout()
        plt.savefig(FIGURES / 'rhetoric_components_2020.png', dpi=150)
        plt.show()